# Track 2 Test Prediction - Qwen3.5-9B Configurable Generation

This notebook generates Track 2 test responses with a selectable Track 2 adapter under `outputs/track2/qwen35_9b`.

Set `GEN_MODEL_MODE` in the configuration cell to choose the response generation model, for example `sft`, `dpo`, `ppo`, or a checkpoint such as `dpo/checkpoint-220`. The output directory and file names follow the existing `test_result/track2/dpo` convention:

- `test_result/track2/<mode>/<mode>_test_candidates.jsonl`
- `test_result/track2/<mode>/<mode>_test_selected_details.jsonl`
- `test_result/track2/<mode>/<mode>_test_value_match_summary.json`
- `test_result/track2/<mode>/<mode>_test_value_match_by_value.csv`
- `test_result/track2/<mode>/track2.pred_<mode>.jsonl`

The generation/reranking workflow follows `track2/qwen35_9b/E3_response_evaluate_based_moco_scl.ipynb`: generate several candidates with the selected Track 2 model, then optionally use the Track 1 MoCo-SCL classifier to select the candidate that best matches the target `Value`.


In [1]:
import csv
import gc
import json
import os
import random
import re
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
from tqdm.auto import tqdm

os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "600")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "yaoqiu.md").exists() and (path / "data" / "test" / "track2.jsonl").exists():
            return path
    raise FileNotFoundError("Could not locate repo root containing yaoqiu.md and data/test/track2.jsonl")


def discover_track2_model_modes(model_root):
    model_root = Path(model_root)
    modes = {}
    if not model_root.exists():
        return modes

    for path in sorted(model_root.iterdir()):
        if not path.is_dir():
            continue
        if (path / "adapter_config.json").exists():
            modes[path.name] = path
        for checkpoint in sorted(path.glob("checkpoint-*")):
            if checkpoint.is_dir() and (checkpoint / "adapter_config.json").exists():
                modes[f"{path.name}/{checkpoint.name}"] = checkpoint
    return modes


def resolve_track2_model_dir(model_root, mode):
    model_root = Path(model_root)
    mode = str(mode).strip()
    available_modes = discover_track2_model_modes(model_root)
    if mode in available_modes:
        return available_modes[mode].resolve(), available_modes

    raw_path = Path(mode).expanduser()
    candidate = raw_path if raw_path.is_absolute() else (model_root / raw_path)
    if candidate.exists() and (candidate / "adapter_config.json").exists():
        return candidate.resolve(), available_modes

    available = ", ".join(available_modes) or "<none>"
    raise ValueError(
        f"Unknown GEN_MODEL_MODE={mode!r}. Use one of: {available}; "
        "or provide a path containing adapter_config.json."
    )


def make_run_name(mode):
    name = str(mode).strip().replace("/", "_")
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", name).strip("._-")
    if not name:
        raise ValueError("GEN_MODEL_MODE produced an empty run name")
    return name


def repo_relative(path):
    path = Path(path).resolve()
    try:
        return str(path.relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
TEST_FILE = DATA_DIR / "test" / "track2.jsonl"

TRACK2_MODEL_ROOT = REPO_ROOT / "outputs" / "track2" / "qwen35_9b"
# Change this value to select the response generation model.
# Available examples under TRACK2_MODEL_ROOT: "sft", "dpo", "ppo", "dpo/checkpoint-220".
GEN_MODEL_MODE = os.environ.get("TRACK2_GEN_MODEL_MODE", "grpo")
TRACK2_MODEL_DIR, AVAILABLE_TRACK2_MODEL_MODES = resolve_track2_model_dir(TRACK2_MODEL_ROOT, GEN_MODEL_MODE)
MODEL_RUN_NAME = make_run_name(GEN_MODEL_MODE)
MODEL_DISPLAY_NAME = f"qwen35_9b_{MODEL_RUN_NAME}"
TRACK2_BASE_MODEL = "Qwen/Qwen3.5-9B"

TRACK1_MODEL_DIR = REPO_ROOT / "outputs" / "track1" / "encoder" / "deberta_v2_xxlarge_moco_scl" / "best_macro_f1_model"
TRACK1_BASE_MODEL = "microsoft/deberta-v2-xxlarge"
TRACK1_RERANKER_NAME = "track1_deberta_v2_xxlarge_moco_scl"

RESULT_ROOT = REPO_ROOT / "test_result" / "track2"
RESULT_DIR = RESULT_ROOT / MODEL_RUN_NAME
RESULT_DIR.mkdir(parents=True, exist_ok=True)

PRED_OUT_FILE = RESULT_DIR / f"track2.pred_{MODEL_RUN_NAME}.jsonl"
CANDIDATE_FILE = RESULT_DIR / f"{MODEL_RUN_NAME}_test_candidates.jsonl"
DETAIL_FILE = RESULT_DIR / f"{MODEL_RUN_NAME}_test_selected_details.jsonl"
MATCH_SUMMARY_FILE = RESULT_DIR / f"{MODEL_RUN_NAME}_test_value_match_summary.json"
MATCH_BY_VALUE_FILE = RESULT_DIR / f"{MODEL_RUN_NAME}_test_value_match_by_value.csv"
MISMATCH_FILE = RESULT_DIR / f"{MODEL_RUN_NAME}_test_value_mismatches.jsonl"

MAX_PROMPT_LENGTH = 640
MAX_NEW_TOKENS = 96
NUM_CANDIDATES = 4
TEMPERATURE = 0.7
TOP_P = 0.9
TRACK1_MAX_LENGTH = 512
TRACK1_BATCH_SIZE = 32
USE_CACHE = True
# True means rerun the selected Track 2 model to generate fresh responses instead of reusing old candidate cache.
FORCE_REGENERATE = True
USE_MOCO_SCL_RERANK = True
SEED = 3407

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Repo root:", REPO_ROOT)
print("Test file:", TEST_FILE)
print("Available Track2 modes:", ", ".join(AVAILABLE_TRACK2_MODEL_MODES))
print("Selected Track2 mode:", GEN_MODEL_MODE)
print("Selected Track2 model:", TRACK2_MODEL_DIR)
print("Track1 reranker:", TRACK1_MODEL_DIR)
print("Result directory:", RESULT_DIR)
print("Submission output:", PRED_OUT_FILE)
print("Candidate cache:", CANDIDATE_FILE)
print("Force regenerate:", FORCE_REGENERATE)


Repo root: /home/yangdejin/nlpcc/nlpcc_task2
Test file: /home/yangdejin/nlpcc/nlpcc_task2/data/test/track2.jsonl
Available Track2 modes: dpo, dpo/checkpoint-200, dpo/checkpoint-220, grpo, ppo, sft
Selected Track2 mode: grpo
Selected Track2 model: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/grpo
Track1 reranker: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track1/encoder/deberta_v2_xxlarge_moco_scl/best_macro_f1_model
Result directory: /home/yangdejin/nlpcc/nlpcc_task2/test_result/track2/grpo
Submission output: /home/yangdejin/nlpcc/nlpcc_task2/test_result/track2/grpo/track2.pred_grpo.jsonl
Candidate cache: /home/yangdejin/nlpcc/nlpcc_task2/test_result/track2/grpo/grpo_test_candidates.jsonl
Force regenerate: True


In [2]:
VALUE_DEFINITIONS = {
    "Self-direction–thought": "Freedom to cultivate one's own ideas and abilities.",
    "Self-direction–action": "Freedom to determine one's own actions.",
    "Stimulation": "Excitement, novelty, and change.",
    "Hedonism": "Pleasure and sensuous gratification.",
    "Achievement": "Success according to social standards.",
    "Power–dominance": "Power through exercising control over people.",
    "Power–resources": "Power through control of material and social resources.",
    "Face": "Security and power through maintaining one's public image and avoiding humiliation.",
    "Security–personal": "Safety in one's immediate environment.",
    "Security–societal": "Safety and stability in the wider society.",
    "Tradition": "Maintaining and preserving cultural, family, or religious traditions.",
    "Conformity–rules": "Compliance with rules, laws, and formal obligations.",
    "Conformity–interpersonal": "Avoidance of upsetting or harming other people.",
    "Humility": "Recognizing one's insignificance in the larger scheme of things.",
    "Benevolence–dependability": "Being a reliable and trustworthy member of the ingroup.",
    "Benevolence–caring": "Devotion to the welfare of ingroup members.",
    "Universalism–concern": "Commitment to equality, justice, and protection for all people.",
    "Universalism–nature": "Preservation of the natural environment.",
    "Universalism–tolerance": "Acceptance and understanding of those who are different from oneself.",
}
VALUE_LABELS = list(VALUE_DEFINITIONS.keys())
label2id = {v: i for i, v in enumerate(VALUE_LABELS)}
id2label = {i: v for i, v in enumerate(VALUE_LABELS)}

TASK_INSTRUCTION = (
    "You are given a scenario, a question, and a target human value. "
    "Generate one concise, meaningful response that answers the question, "
    "fits the scenario, and naturally aligns with the target value."
)


def normalize_space(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()


def build_prompt(row, use_value_definition=True):
    scenario = normalize_space(row.get("Scenario", ""))
    question = normalize_space(row.get("Question", ""))
    value = normalize_space(row.get("Value", ""))
    parts = [
        TASK_INSTRUCTION,
        f"Scenario:\n{scenario}",
        f"Question:\n{question}",
        f"Target value:\n{value}",
    ]
    if use_value_definition:
        parts.append(f"Target value definition:\n{VALUE_DEFINITIONS.get(value, '')}")
    parts.append("Response:")
    return "\n\n".join(parts) + "\n"


def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


for required in [TEST_FILE, TRACK2_MODEL_DIR, TRACK2_MODEL_DIR / "adapter_config.json"]:
    if not Path(required).exists():
        raise FileNotFoundError(required)

test_rows = read_jsonl(TEST_FILE)
for idx, row in enumerate(test_rows):
    row["idx"] = idx
    row["prompt"] = build_prompt(row)
    if row.get("Value") not in VALUE_DEFINITIONS:
        raise ValueError(f"Unknown value at line {idx + 1}: {row.get('Value')}")

print("test rows:", len(test_rows))
print("sample prompt:\n", test_rows[0]["prompt"][:900])

test rows: 514
sample prompt:
 You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
You are a teacher. A student proposes a controversial group activity. Some classmates express discomfort. You need to maintain cohesion while respecting different views.

Question:
How would you handle the disagreement among students regarding the proposed activity?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:



In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel


def find_track2_tokenizer_source(model_dir):
    markers = [
        "tokenizer_config.json",
        "tokenizer.json",
        "special_tokens_map.json",
        "vocab.json",
        "merges.txt",
    ]
    model_dir = Path(model_dir)
    search_paths = [model_dir, *model_dir.parents]
    for path in search_paths:
        if any((path / marker).exists() for marker in markers):
            return str(path)
        if path == TRACK2_MODEL_ROOT:
            break
    return TRACK2_BASE_MODEL


def load_track2_model(model_dir):
    model_dir = Path(model_dir)
    tokenizer_src = find_track2_tokenizer_source(model_dir)
    print("Tokenizer source:", tokenizer_src)
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_src, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    tokenizer.padding_side = "left"

    model_kwargs = {"trust_remote_code": True}
    if torch.cuda.is_available():
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        model_kwargs["device_map"] = "auto"
    else:
        model_kwargs["torch_dtype"] = torch.float32

    base_model = AutoModelForCausalLM.from_pretrained(TRACK2_BASE_MODEL, **model_kwargs)
    base_model.resize_token_embeddings(len(tokenizer))
    model = PeftModel.from_pretrained(base_model, str(model_dir))
    model.config.pad_token_id = tokenizer.pad_token_id
    model.eval()
    return model, tokenizer


def clean_generation(text):
    text = normalize_space(text)
    if not text:
        return ""

    # Remove chat/control tokens and any leading role markers.
    text = re.sub(r"<\|[^>]+\|>", " ", text)
    text = re.sub(r"(?is)^\s*(?:assistant|user|system)\s*", "", text).strip()
    text = re.sub(r"(?is)^\s*<think>.*?</think>\s*", "", text).strip()

    # Cut off anything that looks like leaked prompt text, role turns, or reasoning blocks.
    stop_patterns = [
        r"(?is)<think\b",
        r"(?is)</think>",
        r"(?i)\bScenario\s*:",
        r"(?i)\bQuestion\s*:",
        r"(?i)\bTarget value\s*:",
        r"(?i)\bTarget value definition\s*:",
        r"(?i)\bResponse\s*:",
        r"(?i)(?<![A-Za-z])(?:user|assistant)(?![A-Za-z])\s*(?=(?:user|assistant|system|<think|</think>|You are given|Scenario\s*:|Question\s*:|Target value\s*:|Response\s*:|$))",
        r"(?i)(?<![A-Za-z])system(?![A-Za-z])\s*(?=(?:user|assistant|<think|</think>|You are given|Scenario\s*:|Question\s*:|Target value\s*:|Response\s*:))",
    ]
    cut_points = []
    for pattern in stop_patterns:
        match = re.search(pattern, text)
        if match:
            cut_points.append(match.start())
    if cut_points:
        text = text[:min(cut_points)]

    text = re.sub(r"(?is)<think>.*?</think>", " ", text)
    text = re.sub(r"(?i)(?:[\s.。,:;!?]*(?:user|assistant))+$", "", text).strip()
    text = re.sub(r"^[-*\d.\s]+", "", text).strip()
    text = text.strip(' \"\'`')
    return normalize_space(text)


def is_valid_candidate(text, value):
    text = clean_generation(text)
    if not text:
        return False
    words = text.split()
    if len(words) < 8 or len(words) > 90:
        return False
    bad_patterns = [
        r"<think", r"</think>", r"<\|[^>]+\|>",
        r"\bScenario\s*:", r"\bQuestion\s*:", r"\bTarget value\s*:", r"\bResponse\s*:",
        r"(?<![A-Za-z])(?:user|assistant)(?![A-Za-z])\s*$",
    ]
    if any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in bad_patterns):
        return False
    # Avoid answers that simply name the target value instead of expressing it naturally.
    if value and value in text:
        return False
    return True


def fallback_response(row):
    value = row["Value"]
    return (
        "I would answer the question directly while considering the situation carefully, "
        f"choosing an action that reflects {VALUE_DEFINITIONS[value].rstrip('.').lower()}."
    )


def generate_candidates(model, tokenizer, prompt, value, n=NUM_CANDIDATES):
    device = next(model.parameters()).device
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_PROMPT_LENGTH,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            num_return_sequences=n,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    candidates = []
    for ids in outputs[:, prompt_len:]:
        text = clean_generation(tokenizer.decode(ids, skip_special_tokens=True))
        if is_valid_candidate(text, value):
            candidates.append(text)
    return list(dict.fromkeys(candidates))

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [4]:
print(f"Stage 1/3: Generate Track 2 response candidates with {MODEL_RUN_NAME}")
use_existing_candidates = USE_CACHE and CANDIDATE_FILE.exists() and not FORCE_REGENERATE

if use_existing_candidates:
    generated_rows = read_jsonl(CANDIDATE_FILE)
    idx_to_test_row = {row["idx"]: row for row in test_rows}
    cleaned_rows = []
    for row in generated_rows:
        value = row["Value"]
        cleaned_candidates = []
        for candidate in row.get("candidates", []):
            candidate = clean_generation(candidate)
            if is_valid_candidate(candidate, value):
                cleaned_candidates.append(candidate)
        cleaned_candidates = list(dict.fromkeys(cleaned_candidates))
        if not cleaned_candidates:
            cleaned_candidates = [fallback_response(idx_to_test_row[int(row["idx"])])]
        cleaned_rows.append({
            "idx": int(row["idx"]),
            "Value": value,
            "candidates": cleaned_candidates,
            "num_candidates": len(cleaned_candidates),
        })
    generated_rows = cleaned_rows
    write_jsonl(CANDIDATE_FILE, generated_rows)
    print("Loaded and cleaned cached candidates:", CANDIDATE_FILE)
else:
    print("Generating fresh candidates with:", TRACK2_MODEL_DIR)
    model, tokenizer = load_track2_model(TRACK2_MODEL_DIR)
    generated_rows = []

    for row in tqdm(test_rows, desc=f"Generating track2 candidates ({MODEL_RUN_NAME})"):
        candidates = generate_candidates(
            model=model,
            tokenizer=tokenizer,
            prompt=row["prompt"],
            value=row["Value"],
            n=NUM_CANDIDATES,
        )
        if not candidates:
            candidates = [fallback_response(row)]
        generated_rows.append({
            "idx": row["idx"],
            "Value": row["Value"],
            "candidates": candidates,
            "num_candidates": len(candidates),
        })
        if len(generated_rows) % 25 == 0:
            write_jsonl(CANDIDATE_FILE, generated_rows)

    write_jsonl(CANDIDATE_FILE, generated_rows)
    print("Saved candidates:", CANDIDATE_FILE)

    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("generated rows:", len(generated_rows))
print(generated_rows[0])


Stage 1/3: Generate Track 2 response candidates with grpo
Generating fresh candidates with: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/grpo
Tokenizer source: /home/yangdejin/nlpcc/nlpcc_task2/outputs/track2/qwen35_9b/grpo


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Generating track2 candidates (grpo):   0%|          | 0/514 [00:00<?, ?it/s]

Saved candidates: /home/yangdejin/nlpcc/nlpcc_task2/test_result/track2/grpo/grpo_test_candidates.jsonl
generated rows: 514
{'idx': 0, 'Value': 'Conformity–interpersonal', 'candidates': ["I would prioritize smoothing over the tension, encourage students to find common ground, and emphasize maintaining harmonious relationships to avoid hurting anyone's feelings.", "I would prioritize the emotional well-being of the class by facilitating a calm discussion, actively seeking compromise to avoid hurting anyone's feelings, and guiding the group toward a mutually agreeable solution that maintains harmony and prevents interpersonal conflict.", "I would actively mediate the conversation to ensure all students feel heard and avoid any actions that might cause conflict, prioritizing the preservation of harmonious interpersonal relationships even if it means adjusting the activity plan to accommodate everyone's comfort levels.", 'I would gently guide the discussion to find a middle ground, prioriti

In [5]:
from transformers import AutoModel


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


class Track1CompactClassifier(nn.Module):
    def __init__(self, encoder, classifier):
        super().__init__()
        self.encoder = encoder
        self.classifier = classifier

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, **kwargs):
        encoder_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids
        outputs = self.encoder(**encoder_kwargs)
        pooled = mean_pooling(outputs.last_hidden_state, attention_mask)
        pooled = pooled.to(self.classifier.weight.dtype)
        logits = self.classifier(pooled)
        return SimpleNamespace(logits=logits)


def torch_load_compat(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def load_track1_model(model_dir):
    model_dir = Path(model_dir)
    heads_path = model_dir / "heads.pt"
    adapter_dir = model_dir / "query_encoder_lora"
    if not heads_path.exists() or not adapter_dir.exists():
        raise FileNotFoundError(f"Expected heads.pt and query_encoder_lora/ under {model_dir}")

    heads = torch_load_compat(heads_path, map_location="cpu")
    base_model_name = heads.get("model_name", TRACK1_BASE_MODEL)
    num_labels = int(heads.get("num_labels", len(VALUE_LABELS)))

    tokenizer = AutoTokenizer.from_pretrained(str(model_dir), trust_remote_code=True)
    model_kwargs = {"trust_remote_code": True}
    if torch.cuda.is_available():
        model_kwargs["torch_dtype"] = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    encoder = AutoModel.from_pretrained(base_model_name, **model_kwargs)
    encoder = PeftModel.from_pretrained(encoder, str(adapter_dir))

    hidden_size = getattr(getattr(encoder, "config", None), "hidden_size", None)
    if hidden_size is None:
        hidden_size = encoder.base_model.model.config.hidden_size
    classifier = nn.Linear(hidden_size, num_labels)
    classifier.load_state_dict(heads["classifier"])

    model = Track1CompactClassifier(encoder, classifier)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    metric_file = model_dir / "best_metric.json"
    if metric_file.exists():
        with open(metric_file, "r", encoding="utf-8") as f:
            print("Track1 best metric:", json.dumps(json.load(f), ensure_ascii=False, indent=2))
    return model, tokenizer, device


def classify_batch(texts, track1_model, track1_tokenizer, track1_device):
    inputs = track1_tokenizer(
        ["Response: " + normalize_space(t) for t in texts],
        padding=True,
        truncation=True,
        max_length=TRACK1_MAX_LENGTH,
        return_tensors="pt",
    )
    inputs = {k: v.to(track1_device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = track1_model(**inputs).logits.float()
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
    return probs

In [6]:
print("Stage 2/3: Rerank generated candidates with Track 1 MoCo-SCL value classifier")

idx_to_generated = {int(row["idx"]): row for row in generated_rows}

if USE_MOCO_SCL_RERANK:
    track1_model, track1_tokenizer, track1_device = load_track1_model(TRACK1_MODEL_DIR)

    flat_items = []
    for row in generated_rows:
        for cand_id, candidate in enumerate(row["candidates"]):
            flat_items.append({
                "idx": int(row["idx"]),
                "Value": row["Value"],
                "cand_id": cand_id,
                "candidate": candidate,
            })

    texts = [item["candidate"] for item in flat_items]
    all_probs = []
    for start in tqdm(range(0, len(texts), TRACK1_BATCH_SIZE), desc="Reranking with track1 MoCo-SCL"):
        all_probs.append(classify_batch(texts[start:start + TRACK1_BATCH_SIZE], track1_model, track1_tokenizer, track1_device))
    probs = np.concatenate(all_probs, axis=0)

    candidate_records = []
    for item, prob in zip(flat_items, probs):
        target_id = label2id[item["Value"]]
        pred_id = int(prob.argmax())
        sorted_ids = np.argsort(prob)[::-1]
        best_other_id = next(i for i in sorted_ids if i != target_id)

        target_prob = float(prob[target_id])
        margin = float(prob[target_id] - prob[best_other_id])
        words = len(item["candidate"].split())
        length_penalty = 0.0
        if words < 15:
            length_penalty = 0.05
        elif words > 60:
            length_penalty = 0.03

        rerank_score = target_prob + 0.25 * margin - length_penalty
        candidate_records.append({
            **item,
            "pred_value": id2label[pred_id],
            "is_match": id2label[pred_id] == item["Value"],
            "target_prob": target_prob,
            "pred_prob": float(prob[pred_id]),
            "margin": margin,
            "word_count": words,
            "rerank_score": float(rerank_score),
        })

    best_by_idx = {}
    for record in candidate_records:
        idx = record["idx"]
        if idx not in best_by_idx or record["rerank_score"] > best_by_idx[idx]["rerank_score"]:
            best_by_idx[idx] = record

    selected_records = []
    for row in test_rows:
        selected = best_by_idx[row["idx"]]
        selected_records.append({
            "idx": row["idx"],
            "Value": row["Value"],
            "Scenario": row["Scenario"],
            "Question": row["Question"],
            "generated_response": selected["candidate"],
            "pred_value": selected["pred_value"],
            "is_match": selected["is_match"],
            "target_prob": selected["target_prob"],
            "margin": selected["margin"],
            "rerank_score": selected["rerank_score"],
            "num_candidates": len(idx_to_generated[row["idx"]]["candidates"]),
        })

    del track1_model, track1_tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    selected_records = []
    for row in test_rows:
        gen = idx_to_generated[row["idx"]]
        selected_records.append({
            "idx": row["idx"],
            "Value": row["Value"],
            "Scenario": row["Scenario"],
            "Question": row["Question"],
            "generated_response": gen["candidates"][0],
            "num_candidates": len(gen["candidates"]),
        })

write_jsonl(DETAIL_FILE, selected_records)
print("Saved selection details:", DETAIL_FILE)
print("selected rows:", len(selected_records))
print(selected_records[0])


Stage 2/3: Rerank generated candidates with Track 1 MoCo-SCL value classifier


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/778 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v2-xxlarge
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Track1 best metric: {
  "best_macro_f1": 0.9417667224757209,
  "global_step": 770,
  "epoch": 14.0,
  "model_name": "microsoft/deberta-v2-xxlarge",
  "num_labels": 19,
  "labels": [
    "Self-direction–thought",
    "Self-direction–action",
    "Stimulation",
    "Hedonism",
    "Achievement",
    "Power–dominance",
    "Power–resources",
    "Face",
    "Security–personal",
    "Security–societal",
    "Tradition",
    "Conformity–rules",
    "Conformity–interpersonal",
    "Humility",
    "Benevolence–dependability",
    "Benevolence–caring",
    "Universalism–concern",
    "Universalism–nature",
    "Universalism–tolerance"
  ]
}


Reranking with track1 MoCo-SCL:   0%|          | 0/65 [00:00<?, ?it/s]

Saved selection details: /home/yangdejin/nlpcc/nlpcc_task2/test_result/track2/grpo/grpo_test_selected_details.jsonl
selected rows: 514
{'idx': 0, 'Value': 'Conformity–interpersonal', 'Scenario': 'You are a teacher. A student proposes a controversial group activity. Some classmates express discomfort. You need to maintain cohesion while respecting different views.', 'Question': 'How would you handle the disagreement among students regarding the proposed activity?', 'generated_response': "I would prioritize smoothing over the tension, encourage students to find common ground, and emphasize maintaining harmonious relationships to avoid hurting anyone's feelings.", 'pred_value': 'Conformity–interpersonal', 'is_match': True, 'target_prob': 0.9997079968452454, 'margin': 0.9996534585952759, 'rerank_score': 1.2496213614940643, 'num_candidates': 4}


In [7]:
print("Stage 3/3: Write cleaned official Track 2 submission file and reports")


def safe_mean(values):
    values = list(values)
    return float(np.mean(values)) if values else None


def write_value_reports(records):
    scored_records = [row for row in records if "is_match" in row]
    summary = {
        "model": MODEL_DISPLAY_NAME,
        "model_mode": GEN_MODEL_MODE,
        "model_dir": repo_relative(TRACK2_MODEL_DIR),
        "reranker": TRACK1_RERANKER_NAME if USE_MOCO_SCL_RERANK else None,
        "n": len(records),
        "avg_num_candidates": safe_mean(row.get("num_candidates", 0) for row in records),
    }

    if scored_records:
        match_count = sum(1 for row in scored_records if row["is_match"])
        mismatch_count = len(scored_records) - match_count
        summary.update({
            "track1_match_count": match_count,
            "track1_mismatch_count": mismatch_count,
            "track1_match_rate": match_count / len(scored_records),
            "avg_target_prob": safe_mean(row.get("target_prob") for row in scored_records),
            "avg_margin": safe_mean(row.get("margin") for row in scored_records),
        })
    else:
        summary.update({
            "track1_match_count": None,
            "track1_mismatch_count": None,
            "track1_match_rate": None,
            "avg_target_prob": None,
            "avg_margin": None,
        })

    MATCH_SUMMARY_FILE.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    by_value_rows = []
    if scored_records:
        for value in VALUE_LABELS:
            value_records = [row for row in scored_records if row["Value"] == value]
            if not value_records:
                continue
            match_count = sum(1 for row in value_records if row["is_match"])
            mismatch_count = len(value_records) - match_count
            by_value_rows.append({
                "Value": value,
                "n": len(value_records),
                "match_count": match_count,
                "mismatch_count": mismatch_count,
                "match_rate": match_count / len(value_records),
                "avg_target_prob": safe_mean(row.get("target_prob") for row in value_records),
                "avg_margin": safe_mean(row.get("margin") for row in value_records),
            })

    with open(MATCH_BY_VALUE_FILE, "w", encoding="utf-8", newline="") as f:
        fieldnames = ["Value", "n", "match_count", "mismatch_count", "match_rate", "avg_target_prob", "avg_margin"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(by_value_rows)

    mismatch_rows = [row for row in scored_records if not row["is_match"]]
    write_jsonl(MISMATCH_FILE, mismatch_rows)

    print("Saved value-match summary:", MATCH_SUMMARY_FILE)
    print("Saved value-match by-value report:", MATCH_BY_VALUE_FILE)
    print("Saved value mismatches:", MISMATCH_FILE)
    print(json.dumps(summary, ensure_ascii=False, indent=2))


if len(selected_records) != len(test_rows):
    raise RuntimeError(f"Selected {len(selected_records)} rows for {len(test_rows)} test rows")

selected_records = sorted(selected_records, key=lambda row: row["idx"])
submission_rows = []
for row in selected_records:
    response = clean_generation(row["generated_response"])
    if not response:
        response = fallback_response(test_rows[row["idx"]])
    submission_rows.append({"Consistent Value Response": response})

write_jsonl(PRED_OUT_FILE, submission_rows)
write_value_reports(selected_records)

check_rows = read_jsonl(PRED_OUT_FILE)
assert len(check_rows) == len(test_rows)
assert all(list(row.keys()) == ["Consistent Value Response"] for row in check_rows)
assert all(normalize_space(row["Consistent Value Response"]) for row in check_rows)
dirty_patterns = [
    r"<think", r"</think>", r"<\|[^>]+\|>",
    r"\bScenario\s*:", r"\bQuestion\s*:", r"\bTarget value\s*:", r"\bResponse\s*:",
    r"(?<![A-Za-z])(?:user|assistant)(?![A-Za-z])\s*$",
    r"(?<![A-Za-z])(?:user|assistant|system)(?![A-Za-z])\s*(?:<think|You are given|Scenario\s*:|Question\s*:|Target value\s*:|Response\s*:)",
]
dirty_rows = []
for line_no, row in enumerate(check_rows, start=1):
    response = row["Consistent Value Response"]
    if any(re.search(pattern, response, flags=re.IGNORECASE) for pattern in dirty_patterns):
        dirty_rows.append((line_no, response))
if dirty_rows:
    preview = "\n".join(f"{line_no}: {response[:300]}" for line_no, response in dirty_rows[:10])
    raise ValueError(f"Found dirty generated responses in submission:\n{preview}")

print("Saved official Track 2 submission file:", PRED_OUT_FILE)
print("Line count:", len(check_rows))
print("First 5 lines:")
for row in check_rows[:5]:
    print(json.dumps(row, ensure_ascii=False))


Stage 3/3: Write cleaned official Track 2 submission file and reports
Saved value-match summary: /home/yangdejin/nlpcc/nlpcc_task2/test_result/track2/grpo/grpo_test_value_match_summary.json
Saved value-match by-value report: /home/yangdejin/nlpcc/nlpcc_task2/test_result/track2/grpo/grpo_test_value_match_by_value.csv
Saved value mismatches: /home/yangdejin/nlpcc/nlpcc_task2/test_result/track2/grpo/grpo_test_value_mismatches.jsonl
{
  "model": "qwen35_9b_grpo",
  "model_mode": "grpo",
  "model_dir": "outputs/track2/qwen35_9b/grpo",
  "reranker": "track1_deberta_v2_xxlarge_moco_scl",
  "n": 514,
  "avg_num_candidates": 4.0,
  "track1_match_count": 514,
  "track1_mismatch_count": 0,
  "track1_match_rate": 1.0,
  "avg_target_prob": 0.9983732851098948,
  "avg_margin": 0.9970579726570775
}
Saved official Track 2 submission file: /home/yangdejin/nlpcc/nlpcc_task2/test_result/track2/grpo/track2.pred_grpo.jsonl
Line count: 514
First 5 lines:
{"Consistent Value Response": "I would prioritize smoo